# Maximum Value Per Key — Google Colab runner

**Important GPU note:** The supplied MapReduce framework is a CPU C/pthreads program. Selecting a GPU runtime does not make this C program use the GPU. This notebook checks whether a GPU is available, but runs the existing MapReduce application on CPU. A genuine GPU version would require rewriting the mapper/reducer and data pipeline using CUDA (or another GPU framework).

This notebook mounts Drive, checks the project paths, compiles the C application, runs it with live chunk progress, and saves the output.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Set project and dataset paths

In [ ]:
import os
from pathlib import Path

project = "/content/drive/MyDrive/C project"
framework = os.path.join(project, "mapreduce_framework_v2")
examples = os.path.join(framework, "examples")
src = os.path.join(framework, "src")
dataset = os.path.join(project, "datasets", "Max_val_per_key")

c_file = os.path.join(examples, "max_per_key_folder_optimized.c")
executable = os.path.join(examples, "max_per_key_folder")

for label, path in [
    ("Framework", framework),
    ("Examples", examples),
    ("Source directory", src),
    ("Dataset", dataset),
    ("Optimized C file", c_file),
]:
    print(f"{label}: {'OK' if os.path.exists(path) else 'MISSING'} — {path}")

assert os.path.isdir(framework), "Framework folder not found. Check project path."
assert os.path.isdir(src), "src folder not found."
assert os.path.isdir(dataset), "Dataset folder not found."
assert os.path.isfile(c_file), (
    "Upload max_per_key_folder_optimized.c into the framework examples folder."
)

Framework: OK — /content/drive/MyDrive/C project/mapreduce_framework_v2
Examples: OK — /content/drive/MyDrive/C project/mapreduce_framework_v2/examples
Source directory: OK — /content/drive/MyDrive/C project/mapreduce_framework_v2/src
Dataset: OK — /content/drive/MyDrive/C project/datasets/Max_val_per_key
Optimized C file: OK — /content/drive/MyDrive/C project/mapreduce_framework_v2/examples/max_per_key_folder_optimized.c


## 3. Check dataset files and total size

In [ ]:
csv_files = sorted(
    f for f in os.listdir(dataset)
    if f.lower().endswith(".csv")
)

total_bytes = sum(os.path.getsize(os.path.join(dataset, f)) for f in csv_files)

print("CSV files:", len(csv_files))
print("Total size: %.2f GB" % (total_bytes / (1024**3)))
for f in csv_files:
    size_mb = os.path.getsize(os.path.join(dataset, f)) / (1024**2)
    print(f"{f}: {size_mb:.2f} MB")

assert len(csv_files) == 20, f"Expected 20 CSV files, found {len(csv_files)}"

CSV files: 20
Total size: 3.23 GB
2007.csv: 168.88 MB
2008.csv: 176.40 MB
2009.csv: 179.53 MB
2010.csv: 180.80 MB
2011.csv: 171.65 MB
2012.csv: 169.21 MB
2013.csv: 164.19 MB
2014.csv: 162.73 MB
2015.csv: 162.67 MB
2016.csv: 163.37 MB
2017.csv: 163.31 MB
2018.csv: 164.20 MB
2019.csv: 163.78 MB
2020.csv: 164.46 MB
2021.csv: 167.28 MB
2022.csv: 167.56 MB
2023.csv: 167.93 MB
2024.csv: 168.43 MB
2025.csv: 158.41 MB
2026.csv: 118.86 MB


## 4. Check runtime and install CPU build tools

In [ ]:
import subprocess

gpu_check = subprocess.run(
    ["bash", "-lc", "nvidia-smi"],
    capture_output=True, text=True
)
if gpu_check.returncode == 0:
    print("GPU is available in this Colab runtime:")
    print(gpu_check.stdout[:1500])
else:
    print("No GPU detected. The current C MapReduce program runs on CPU either way.")

install = subprocess.run(
    ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq build-essential zlib1g-dev"],
    capture_output=True, text=True
)
print(install.stdout[-1000:])
print(install.stderr[-1000:])
if install.returncode != 0:
    raise RuntimeError("Could not install compiler/zlib dependencies.")

GPU is available in this Colab runtime:
Tue Nov 10 19:10:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------

## 5. Compile the C MapReduce application

In [ ]:
import glob
import subprocess

framework_sources = sorted(glob.glob(os.path.join(src, "*.c")))
assert framework_sources, "No .c framework source files found in src/."

compile_command = [
    "gcc",
    "-O2",
    "-std=c11",
    "-D_POSIX_C_SOURCE=200809L",
    "-I" + os.path.join(framework, "include"),
    "-I" + src,
    "-I" + examples,
    c_file,
    *framework_sources,
    "-o", executable,
    "-pthread",
    "-lz",
    "-lm",
]

print("Compiling", len(framework_sources), "framework source files...")
build = subprocess.run(compile_command, capture_output=True, text=True)

if build.stdout:
    print(build.stdout)
if build.stderr:
    print(build.stderr)

if build.returncode != 0:
    raise RuntimeError("Compilation failed. Review the compiler errors above.")

print("Compilation successful:", executable)

Compiling 9 framework source files...
Compilation successful: /content/drive/MyDrive/C project/mapreduce_framework_v2/examples/max_per_key_folder


## 6. Run with live progress

In [ ]:
import time
import subprocess

run_command = [executable, dataset]
print("Starting MapReduce on", len(csv_files), "CSV files")
print("This is a CPU MapReduce run; a GPU runtime does not accelerate this binary.")
print("-" * 65)

start_time = time.time()
output_lines = []

process = subprocess.Popen(
    run_command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    process.wait()
except KeyboardInterrupt:
    process.terminate()
    process.wait()
    print("\nExecution interrupted by user.")

elapsed = time.time() - start_time
print("\n" + "=" * 65)
print("Exit code:", process.returncode)
print("Elapsed time: %.2f seconds" % elapsed)
print("=" * 65)

if process.returncode != 0:
    print("The program did not finish successfully. Check the error output above.")

Streaming output truncated to the last 5000 lines.
US1WYLM0243      MDPR     406
US1UTWG0022      MDPR     5
US1ALLS0022      WESF     91
USC00295960      TMAX     361
US1COBO0277      PRCP     417
US1CATH0005      SNWD     127
US1ORBN0125      SNWD     0
USC00245387      SNWD     483
US1COEP0454      PRCP     711
BF1SW000003      PRCP     655
ASN00015618      PRCP     1460
USW00024222      TMIN     228
US1COWE0315      PRCP     826
MXN00015030      TMAX     300
RQW00011630      WT13     1
US1COAR0327      MDPR     483
USC00413060      MNPN     339
US1ILDP0156      PRCP     721
USC00411430      WT11     1
US1PAMT0038      WESD     117
US1WAIS0035      WESF     94
US1SCRC0080      PRCP     841
US1KSSU0005      SNOW     0
US1NMLA0007      WESF     206
CAN04012130      PRCP     260
US1KSEW0047      DAPR     4
US1MNHN0648      PRCP     701
US1ILCP0105      SNOW     10
GMW00035032      TAVG     321
US1RIWS0016      PRCP     1280
USC00477115      SNWD     1041
ASN00088001      MDPR     820
M

## 7. Save output to Google Drive

In [ ]:
output_file = os.path.join(project, "max_per_key_output.txt")

with open(output_file, "w") as f:
    f.writelines(output_lines)

print("Output saved to:", output_file)

Output saved to: /content/drive/MyDrive/C project/max_per_key_output.txt


## Notes

- The C source expects the GHCN-Daily field order `ID,DATE,ELEMENT,DATA_VALUE,M_FLAG,Q_FLAG,...`; it does not use `OBS_TIME`.
- The current source skips rows with a non-empty `Q_FLAG` and the `-9999` missing-value sentinel.
- The current implementation processes CSV files sequentially and runs MapReduce chunks within each file.
- Do not run the same executable twice against the same dataset at the same time.
- If compilation fails, share the complete compiler error. If execution fails, share the last progress lines and exit code.